In [ ]:
!pip install -U transformers datasets scikit-learn

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "en_train.json",
        "test": "en_test.json"
    }
)

print(dataset)
print("Ví dụ tập train:", dataset["train"][0])
print("Ví dụ tập test:", dataset["test"][0])

In [ ]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {v: k for k, v in label2id.items()}

def encode_label(example):
    example["label"] = label2id[example["sentiment"]]
    return example

# Áp dụng map cho toàn bộ DatasetDict (cả train và test đều sẽ được xử lý)
dataset = dataset.map(encode_label)

In [ ]:
# Tách tập train sẵn có thành train_dataset và val_dataset (ví dụ: 10% cho validation)
train_val_split = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = train_val_split["train"]
val_dataset = train_val_split["test"]

# Tập test được lấy nguyên vẹn từ file en_test.json
test_dataset = dataset["test"]

print(f"Số lượng - Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [ ]:
def tokenize(example):
    return tokenizer(
        example["sentence"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta-financial",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=200,

    save_total_limit=2,

    fp16=True,

    report_to="none"
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    # Tính toán precision, recall, f1 dạng macro
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1,               # Trả về để khớp với metric_for_best_model="f1" của bạn
        "macro_f1": f1,         # Xuất thêm key hiển thị rõ ràng "macro_f1"
        "precision": precision,
        "recall": recall
    }

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.001
        )
    ]
)

trainer.train()

In [ ]:
# Sử dụng predict cho tập test độc lập để ép hệ thống tính toán lại toàn bộ runtime
prediction_output = trainer.predict(test_dataset)
results = prediction_output.metrics

# Lấy thông tin epoch thực tế từ trạng thái Trainer
actual_epoch = trainer.state.epoch if trainer.state.epoch is not None else 5.0

print("Kết quả đánh giá trên tập Evaluation:")
print(f"eval_loss: {results.get('test_loss', 0):.4f}")
print(f"eval_accuracy: {results.get('test_accuracy', 0):.4f}")
print(f"eval_f1: {results.get('test_f1', 0):.4f}")
print(f"eval_macro_f1: {results.get('test_macro_f1', 0):.4f}")
print(f"eval_precision: {results.get('test_precision', 0):.4f}")
print(f"eval_recall: {results.get('test_recall', 0):.4f}")
print(f"eval_runtime: {results.get('test_runtime', 0):.4f}")
print(f"eval_samples_per_second: {results.get('test_samples_per_second', 0):.4f}")
print(f"eval_steps_per_second: {results.get('test_steps_per_second', 0):.4f}")
print(f"epoch: {actual_epoch:.4f}")